# AI CAD Converter on Google Colab

วิธีนี้ไม่ต้องมี GitHub: อัปโหลด ZIP ของโปรแกรม, ติดตั้งไลบรารี, อัปโหลดแบบ, ดู Preview และค่อยดาวน์โหลดผลลัพธ์ในเซลล์สุดท้าย

In [ ]:
from google.colab import files

uploaded = files.upload()  # เลือกไฟล์ ZIP ของโปรแกรมที่ดาวน์โหลดมา
zip_files = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_files:
    raise ValueError('กรุณาอัปโหลดไฟล์ ZIP ของโปรแกรม')
SOURCE_ARCHIVE = zip_files[0]
print('Uploaded:', SOURCE_ARCHIVE)

In [ ]:
from pathlib import Path
import os
import zipfile

PROJECT_DIR = Path('/content/ai-cad-converter')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(SOURCE_ARCHIVE) as archive:
    archive.extractall(PROJECT_DIR)
os.chdir(PROJECT_DIR)
print('Project folder:', PROJECT_DIR)

In [ ]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq tesseract-ocr tesseract-ocr-tha
!pip install -q -r requirements.txt
!tesseract --list-langs

## อัปโหลดรูปภาพหรือ PDF ที่ต้องการแปลง

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
allowed = {'.pdf', '.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp', '.webp'}
inputs = [name for name in uploaded if Path(name).suffix.lower() in allowed]
if not inputs:
    raise ValueError('กรุณาอัปโหลด PDF หรือไฟล์รูปภาพที่รองรับ')
INPUT_FILE = str(Path(inputs[0]).resolve())
print('Input:', INPUT_FILE)

## แปลงและดู Preview ก่อนดาวน์โหลด

In [ ]:
from IPython.display import Image, display, Markdown
from cad_converter import CADConverter, ConversionConfig
from cad_converter.archive import create_output_archive

OUTPUT_DIR = Path('/content/ai-cad-output')
config = ConversionConfig(
    ocr_languages='tha+eng',
    pdf_dpi=300,
    max_iterations=3,
    desired_score=0.92,
    export_dwg=False,  # Colab สร้าง DXF; ใช้ AutoCAD/GstarCAD Save As เป็น DWG ได้
)
converter = CADConverter(config=config, feedback_path='/content/ai-cad-feedback.jsonl')
result = converter.convert(INPUT_FILE, OUTPUT_DIR)

for page in result.pages:
    score = page.candidate.metrics.final_score * 100
    display(Markdown(f'### หน้า {page.page_number}: QA {score:.1f}%'))
    display(Image(filename=str(page.preview_path)))
    print('DXF:', page.dxf_path.name)

ARCHIVE_PATH = create_output_archive(OUTPUT_DIR, '/content/AI_CAD_Converter_results.zip')
print('ตรวจ Preview เสร็จแล้ว ให้รันเซลล์ถัดไปเมื่อต้องการดาวน์โหลด')

## ดาวน์โหลดผลลัพธ์หลังตรวจ Preview

In [ ]:
from google.colab import files
files.download(str(ARCHIVE_PATH))